# DCRB-Net — Reproducible Colorectal Polyp Segmentation Experiment (v0.1)

**Final paper title:** *DCRB-Net: Domain-Consistency and Reliability-Guided Boundary Fusion for Cross-Dataset Colorectal Polyp Segmentation*.

**Target:** SN Computer Science (Springer Nature). **Primary reference:** Araujo Junior et al. (2025), [DOI:10.1007/s42979-025-04177-w](https://doi.org/10.1007/s42979-025-04177-w). **Companion:** Sushama & Menon (2024), [DOI:10.1007/s42979-024-02932-z](https://doi.org/10.1007/s42979-024-02932-z).

> **Research integrity / replication status:** The primary and companion articles are subscription-only at their official publisher pages, and their detailed protocols/code have **not** been verified. This notebook implements **reproducible generic U-Net controls plus an original experimental DCRB-Net prototype**, **not** a claim to reproduce either published method or their reported numbers. True paper-to-paper reproduction requires full methods, parameter settings, exact splits, and any available implementation.

**Resume guarantee:** atomic latest-checkpoint saving after every `CHECKPOINT_EVERY_BATCHES` completed batches and at every completed epoch; restores network, optimizer, scheduler, AMP scaler, seed states, early stopping state, epoch, ordered sample indices, and next batch. At worst, work since the last checkpoint must be repeated. No mechanism can prevent loss of an interrupted in-flight batch. Local checkpoint files must be trusted before loading.

**Data:** manually download according to each dataset's terms; do not mix an external test set into training or model selection. Notebook defaults to *non-training* smoke checks, so it can be opened and tested without datasets. Set `RUN_EXPERIMENTS=True` when data are ready.

## 1. Environment and reproducibility

Required packages (install into the selected VS Code/Jupyter environment):

`pip install torch torchvision pillow numpy pandas matplotlib nbformat`

Run training only when CUDA is available if desired; the notebook prints whether it is detected. No download is written to the system drive by this notebook.

**CUDA required:** Use the dedicated D: virtual environment and `START_VSCODE_DCRB.cmd`. All experiment outputs and caches must remain under D:.


In [1]:
import os
from pathlib import Path as _DPath
_D_PROJECT = _DPath('D:/RISETKOLABORASI/1 DCRB-Net/EXPERIMEN')
_D_CACHE = _D_PROJECT / 'cache'
for _sub in ['tmp','torch','huggingface','pip','matplotlib','ipython','jupyter','pycache','numba']:
    (_D_CACHE/_sub).mkdir(parents=True,exist_ok=True)
_D_ENV = {'TEMP': 'tmp','TMP': 'tmp','TMPDIR':'tmp','TORCH_HOME':'torch','HF_HOME':'huggingface','PIP_CACHE_DIR':'pip','XDG_CACHE_HOME':'tmp','MPLCONFIGDIR':'matplotlib','IPYTHONDIR':'ipython','JUPYTER_DATA_DIR':'jupyter','JUPYTER_RUNTIME_DIR':'jupyter','NUMBA_CACHE_DIR':'numba','PYTHONPYCACHEPREFIX':'pycache'}
for _key,_sub in _D_ENV.items(): os.environ[_key]=str(_D_CACHE/_sub)
os.environ['POLYP_PROJECT_ROOT']=str(_D_PROJECT)

import os, sys, math, json, hashlib, random, time, copy
from pathlib import Path
from dataclasses import dataclass, asdict
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

print('Python:', sys.version.split()[0], '| PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0), '| CUDA:', torch.version.cuda)
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required. Select D:/.venv kernel and verify CUDA-enabled PyTorch before running this notebook.')


Python: 3.13.5 | PyTorch: 2.10.0+cpu
CUDA available: False


### 1.1 Experiment configuration

**Expected directory layout** (replace root with your actual D: dataset folder):

```text
D:/RISETKOLABORASI/1 DCRB-Net/EXPERIMEN/
  data/
    Kvasir-SEG/images/ and masks/
    CVC-ClinicDB/images/ and masks/
    ETIS-Larib/images/ and masks/
  outputs/  # created by notebook
```

Dataset links: [Kvasir-SEG](https://datasets.simula.no/kvasir-seg/), [CVC-ClinicDB](https://polyp.grand-challenge.org/CVCClinicDB/), [ETIS](https://polyp.grand-challenge.org/ETISLarib/). In actual archives, names/structure may differ: normalize manually before executing. **Never invent missing masks or sample identities**.

In [2]:
@dataclass(frozen=True)
class Config:
    project_root: str = os.getenv('POLYP_PROJECT_ROOT', 'D:/RISETKOLABORASI/1 DCRB-Net/EXPERIMEN')
    image_size: int = 256
    batch_size: int = 8
    lr: float = 2e-4
    weight_decay: float = 1e-4
    max_epochs: int = 60
    early_stop_patience: int = 10
    checkpoint_every_batches: int = 10
    consistency_weight: float = 0.10
    boundary_weight: float = 0.20
    consistency_every_batches: int = 4  # lower compute cost
    train_fraction: float = 0.8
    split_seed: int = 2025
    num_workers: int = 0  # crucial for exact mid-epoch resume with this implementation
    base_channels: int = 24

CFG = Config()
PROJECT = Path(CFG.project_root)
DATA_ROOT = PROJECT / 'data'
OUTPUT_ROOT = PROJECT / 'outputs'
RUN_EXPERIMENTS = False  # set True only after putting downloaded datasets in DATA_ROOT
SEEDS = [42, 123, 2025]  # required final multi-seed evaluation
TRAIN_SOURCES = ['Kvasir-SEG', 'CVC-ClinicDB']
EXTERNAL_TEST_SOURCES = ['ETIS-Larib']
EXPERIMENTS = [
    {'name':'UNet_Kvasir_only', 'architecture':'unet', 'sources':['Kvasir-SEG']},
    {'name':'UNet_combined', 'architecture':'unet', 'sources':TRAIN_SOURCES},
    {'name':'DCRBNet_combined', 'architecture':'dcrb', 'sources':TRAIN_SOURCES},
]
assert CFG.num_workers == 0, 'Mid-epoch reproducibility requires num_workers=0.'
print('Experiment writes to:', OUTPUT_ROOT)
print('Mode:', 'FULL TRAINING' if RUN_EXPERIMENTS else 'MODEL-SMOKE-ONLY')

Experiment writes to: D:/RISETKOLABORASI/POLYP_SEGMENTATION/outputs
Mode: MODEL-SMOKE-ONLY


## 2. Dataset manifest, split isolation, and leakage check

Official datasets may contain different naming conventions; the loader **requires explicit matching image/mask stems**. Per-source 80:20 stratified-by-source training/validation split is a **new protocol** rather than an assertion of the primary paper's split. External test data must never influence checkpoints or hyperparameter choices. Also assess patient/video-level independence where metadata permit; byte hashes alone cannot guarantee it.

In [3]:
IMAGE_EXTS = {'.png','.jpg','.jpeg','.bmp','.tif','.tiff'}

def files_by_stem(folder):
    if not folder.is_dir():
        raise FileNotFoundError(f'Missing required directory: {folder}')
    found = {}
    for p in sorted(folder.iterdir()):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            if p.stem in found:
                raise ValueError(f'Duplicate sample stem in {folder}: {p.stem}')
            found[p.stem] = p
    return found

def source_rows(source):
    base = DATA_ROOT / source
    images = files_by_stem(base/'images')
    masks  = files_by_stem(base/'masks')
    if not images or images.keys() != masks.keys():
        raise ValueError(f'{source}: image/mask stems missing or different: images={len(images)}, masks={len(masks)}; ensure exact pairing')
    return [{'source':source, 'sample_id':f'{source}/{stem}', 'image':str(images[stem]), 'mask':str(masks[stem])}
            for stem in sorted(images)]

def build_manifest():
    all_sources = list(dict.fromkeys(TRAIN_SOURCES + EXTERNAL_TEST_SOURCES))
    result = pd.DataFrame([r for s in all_sources for r in source_rows(s)])
    if result.sample_id.duplicated().any():
        raise ValueError('Duplicate sample identifiers')
    return result

def split_manifest(full):
    rng = np.random.default_rng(CFG.split_seed)
    frames=[]
    for source in TRAIN_SOURCES:
        sub = full.loc[full.source.eq(source)].copy().sort_values('sample_id')
        indices = rng.permutation(len(sub))
        n = max(1, min(len(sub)-1, int(len(sub)*CFG.train_fraction)))
        sub['split']='validation'
        sub.loc[sub.index[indices[:n]],'split']='train'
        frames.append(sub)
    for source in EXTERNAL_TEST_SOURCES:
        sub=full.loc[full.source.eq(source)].copy()
        sub['split']='external_test'
        frames.append(sub)
    return pd.concat(frames,ignore_index=True)

def file_sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda: f.read(1024*1024), b''):
            h.update(b)
    return h.hexdigest()

def check_exact_duplicate_leakage(manifest):
    # Exact-byte duplicate check, NOT patient/video-level or perceptual duplicate screening.
    seen={}
    for row in manifest.itertuples():
        h=file_sha256(row.image)
        if h in seen and seen[h][0] != row.split:
            raise ValueError(f'Cross-split exact duplicate images: {seen[h]} vs {(row.split,row.sample_id)}. Deduplicate/re-split first.')
        seen[h] = (row.split,row.sample_id)
    print(f'Exact-file duplicate leakage check passed: {len(manifest)} entries (patient-level still unverified).')

def manifest_signature(df):
    stable='\n'.join('|'.join(map(str, rec)) for rec in df[['sample_id','image','mask','split']].sort_values('sample_id').itertuples(index=False,name=None))
    return hashlib.sha256(stable.encode()).hexdigest()

manifest = None
if RUN_EXPERIMENTS:
    manifest=split_manifest(build_manifest())
    check_exact_duplicate_leakage(manifest)
    OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
    manifest.to_csv(OUTPUT_ROOT/'dataset_manifest.csv',index=False)
    print(manifest.groupby(['source','split']).size())

## 3. Data loader and model architecture

### Domain-Consistency and Reliability-Guided Boundary Fusion (prototype)

In skip feature $F_l$, instance-normalized content and local-context disagreement generate a spatial reliability gate. **Source-domain running-statistic dispersion** ($V_l$) is accumulated using only the *training-source labels* (Kvasir-SEG / CVC-ClinicDB), then attenuates the reliability of domain-unstable channels. High-frequency contrast generates a separate boundary gate. A separate edge head is trained with boundary targets. During optimization, a style-perturbed second view adds prediction-consistency regularization. No ETIS images/labels/statistics are used to calibrate the gates. **Scientific distinction to test**: source-only domain-generalization (DG), not target-aware unsupervised domain adaptation (UDA). Existing boundary-aware and prototype-based systems already exist; the novelty of this *particular source-dispersion-calibrated skip mechanism* is a hypothesis requiring ablation, literature checks, and external validation, not established SOTA.

In [4]:
class PolypDataset(Dataset):
    def __init__(self, table, image_size):
        self.rows=table.reset_index(drop=True)
        self.sz=image_size
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        row=self.rows.iloc[i]
        with Image.open(row.image) as im:
            im=im.convert('RGB').resize((self.sz,self.sz),Image.Resampling.BILINEAR)
            x=np.asarray(im,dtype=np.float32).transpose(2,0,1)/255.
        with Image.open(row['mask']) as im:
            im=im.convert('L').resize((self.sz,self.sz),Image.Resampling.NEAREST)
            y=(np.asarray(im,dtype=np.float32)>=127).astype(np.float32)[None,:,:]
        return torch.from_numpy(x.copy()),torch.from_numpy(y.copy()),torch.tensor(TRAIN_SOURCES.index(row.source) if row.source in TRAIN_SOURCES else -1,dtype=torch.long)

class DoubleConv(nn.Module):
    def __init__(self,in_c,out_c):
        super().__init__()
        self.block=nn.Sequential(nn.Conv2d(in_c,out_c,3,padding=1,bias=False),nn.BatchNorm2d(out_c),nn.ReLU(inplace=True),
                                 nn.Conv2d(out_c,out_c,3,padding=1,bias=False),nn.BatchNorm2d(out_c),nn.ReLU(inplace=True))
    def forward(self,x):return self.block(x)

class ReliabilityBoundarySkip(nn.Module):
    """Source-only channel-dispersion calibrated skip fusion; no target adaptation."""
    def __init__(self,c,n_sources=2,ema=0.05):
        super().__init__()
        self.ema=ema
        self.register_buffer('source_means',torch.zeros(n_sources,c))
        self.register_buffer('source_seen',torch.zeros(n_sources,dtype=torch.long))
        self.reliability=nn.Sequential(nn.Conv2d(c*4,c,1),nn.ReLU(inplace=True),nn.Conv2d(c,c,1),nn.Sigmoid())
        self.boundary=nn.Sequential(nn.Conv2d(c,c,3,padding=1),nn.Sigmoid())
        self.out=nn.Conv2d(c,c,1)

    @torch.no_grad()
    def update_source_stats(self,skip,source_ids):
        if source_ids is None:return
        summaries=skip.detach().mean(dim=(2,3))
        for sid in source_ids.unique().tolist():
            if sid<0 or sid>=self.source_means.size(0):continue
            avg=summaries[source_ids==sid].mean(0)
            if self.source_seen[sid]==0:self.source_means[sid].copy_(avg)
            else:self.source_means[sid].lerp_(avg,self.ema)
            self.source_seen[sid]+=1

    def forward(self,skip,source_ids=None,update_stats=False):
        if self.training and update_stats:self.update_source_stats(skip,source_ids)
        content=F.instance_norm(skip)
        context=F.avg_pool2d(content,3,stride=1,padding=1)
        inconsistency=(content-context).abs()
        appearance=(skip-content).abs()
        if self.source_seen.ge(1).sum()>=2:
            means=self.source_means[self.source_seen.ge(1)]
            var=means.var(dim=0,unbiased=False)
            dispersion=(var/(var.mean()+1e-6)).clamp(0,5)
        else:
            dispersion=torch.zeros_like(self.source_means[0])
        dispersion=dispersion.view(1,-1,1,1).expand_as(skip)
        # Stable across source domains => more trustworthy; unstable channels attenuated.
        reliability=self.reliability(torch.cat([content,inconsistency,appearance,dispersion],dim=1))
        channel_stability=1.0/(1.0+dispersion)
        reliable=reliability*channel_stability
        bound=self.boundary(inconsistency)
        return self.out(skip*(0.5+0.5*reliable) + content*bound*0.5)

class SegmentationCNN(nn.Module):
    def __init__(self,base=24,proposed=False):
        super().__init__()
        self.proposed=proposed
        self.enc1=DoubleConv(3,base)
        self.enc2=DoubleConv(base,base*2)
        self.enc3=DoubleConv(base*2,base*4)
        self.bridge=DoubleConv(base*4,base*8)
        self.gates=nn.ModuleList([ReliabilityBoundarySkip(c,n_sources=len(TRAIN_SOURCES)) for c in [base,base*2,base*4]]) if proposed else None
        self.dec3=DoubleConv(base*8+base*4,base*4)
        self.dec2=DoubleConv(base*4+base*2,base*2)
        self.dec1=DoubleConv(base*2+base,base)
        self.mask_head=nn.Conv2d(base,1,1)
        self.edge_head=nn.Conv2d(base,1,1) if proposed else None
    def forward(self,x,source_ids=None,update_domain_stats=False):
        s1=self.enc1(x)
        s2=self.enc2(F.max_pool2d(s1,2))
        s3=self.enc3(F.max_pool2d(s2,2))
        mid=self.bridge(F.max_pool2d(s3,2))
        if self.proposed:
            s1,s2,s3=(gate(feat,source_ids,update_domain_stats) for gate,feat in zip(self.gates,(s1,s2,s3)))
        d3=self.dec3(torch.cat([F.interpolate(mid,size=s3.shape[-2:],mode='bilinear',align_corners=False),s3],1))
        d2=self.dec2(torch.cat([F.interpolate(d3,size=s2.shape[-2:],mode='bilinear',align_corners=False),s2],1))
        d1=self.dec1(torch.cat([F.interpolate(d2,size=s1.shape[-2:],mode='bilinear',align_corners=False),s1],1))
        return self.mask_head(d1), (self.edge_head(d1) if self.proposed else None)

# Actual architecture smoke test without datasets or training
for name, prop in [('U-Net',False),('DCRB-Net',True)]:
    with torch.no_grad():
        m=SegmentationCNN(base=8,proposed=prop).eval()
        z,e=m(torch.rand(1,3,64,64))
        assert z.shape==(1,1,64,64)
        assert (e is None) if not prop else e.shape==(1,1,64,64)
    print('PASS:',name,'forward dimensions')

PASS: U-Net forward dimensions
PASS: DCRB-Net forward dimensions


## 4. Optimization objective, metrics, and reliable checkpoints

Loss: $L = L_{BCE}+L_{softDice}+\lambda_b L_{boundary}+\lambda_c L_{style-consistency}$. The consistency loss regularizes outputs under photometric style shifts, using a detached teacher prediction on the first view. **This consistency term is a training regularizer, not evidence that target domains were seen during training.**

Mean Dice and IoU are computed per image, then averaged. Also export per-source external test scores and confidence intervals when final multi-seed experiments are complete.

In [5]:
def set_global_seeds(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=False
    torch.backends.cudnn.deterministic=True

def augment_aligned(x,y):
    # Global RNG state is included in checkpoints; no DataLoader worker RNG.
    if torch.rand((),device=x.device)<0.5: x,y=x.flip(-1),y.flip(-1)
    if torch.rand((),device=x.device)<0.5: x,y=x.flip(-2),y.flip(-2)
    return x,y

def style_shift(x):
    scale=(0.75+0.5*torch.rand((x.size(0),1,1,1),device=x.device))
    bias=(-0.12+0.24*torch.rand((x.size(0),1,1,1),device=x.device))
    return (x*scale+bias).clamp(0.,1.)

def soft_dice_loss(logits,mask,eps=1e-6):
    p=torch.sigmoid(logits)
    num=2*(p*mask).sum((1,2,3))+eps
    den=p.sum((1,2,3))+mask.sum((1,2,3))+eps
    return (1-num/den).mean()

def boundary_mask(mask):
    dil=F.max_pool2d(mask,3,stride=1,padding=1)
    ero=1-F.max_pool2d(1-mask,3,stride=1,padding=1)
    return (dil-ero).clamp(0,1)

def segmentation_loss(logits,mask):
    return F.binary_cross_entropy_with_logits(logits,mask)+soft_dice_loss(logits,mask)

@torch.no_grad()
def evaluate_model(model,dataframe,return_rows=False):
    model.eval()
    dataset=PolypDataset(dataframe,CFG.image_size)
    loader=DataLoader(dataset,batch_size=CFG.batch_size,shuffle=False,num_workers=0)
    result=[];cursor=0
    for x,y,_ in loader:
        x=x.to(DEVICE,non_blocking=True)
        with torch.autocast(device_type='cuda',enabled=(DEVICE.type=='cuda')):
            logits,_=model(x)
        pred=(torch.sigmoid(logits).float().cpu()>0.5).float()
        intersection=(pred*y).sum((1,2,3)); a=pred.sum((1,2,3)); b=y.sum((1,2,3))
        dice=((2*intersection+1e-6)/(a+b+1e-6)).numpy()
        iou=((intersection+1e-6)/(a+b-intersection+1e-6)).numpy()
        for j in range(len(dice)):
            row=dataframe.iloc[cursor+j]
            result.append({'sample_id':row.sample_id,'source':row.source,'split':row.split,'dice':float(dice[j]),'iou':float(iou[j])})
        cursor+=len(dice)
    if not result:raise ValueError('No evaluation samples')
    scores=pd.DataFrame(result)
    return scores if return_rows else scores[['dice','iou']].mean().to_dict()

def capture_rng():
    return {'python':random.getstate(),'numpy':np.random.get_state(),
            'torch':torch.get_rng_state(),'cuda':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}

def restore_rng(state):
    random.setstate(state['python']); np.random.set_state(state['numpy']); torch.set_rng_state(state['torch'])
    if torch.cuda.is_available() and state['cuda'] is not None:torch.cuda.set_rng_state_all(state['cuda'])

def save_atomic(path,state):
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    torch.save(state,tmp)
    os.replace(tmp,path)

print('Loss/metric and checkpoint utilities loaded')

Loss/metric and checkpoint utilities loaded


### 4.1 Auto-resume at the next unprocessed mini-batch

Restoration is **exact from a completed checkpoint boundary** on the same environment/seed/config/data ordering (allowing for hardware/operator nondeterminism). It never claims to recover an interrupted mini-batch that was not checkpointed. For final reporting, use an independent test evaluation only after model selection on validation data.

In [6]:
def train_one(spec, seed, full_manifest):
    train_table=full_manifest.loc[full_manifest.split.eq('train') & full_manifest.source.isin(spec['sources'])].reset_index(drop=True)
    val_table=full_manifest.loc[full_manifest.split.eq('validation') & full_manifest.source.isin(spec['sources'])].reset_index(drop=True)
    if len(train_table)==0 or len(val_table)==0:raise ValueError('Empty source-specific train/validation split')
    manifest_id=manifest_signature(pd.concat([train_table,val_table]))
    set_global_seeds(seed)
    run_name=f"{spec['name']}_seed{seed}"
    run_dir=OUTPUT_ROOT/run_name
    run_dir.mkdir(parents=True,exist_ok=True)
    latest=run_dir/'checkpoint_latest.pt'; best=run_dir/'checkpoint_best.pt'
    history_file=run_dir/'history.jsonl'
    proposed=(spec['architecture']=='dcrb')
    if spec['architecture'] not in ('unet','dcrb'):
        raise NotImplementedError('Only explicitly implemented architectures may be benchmarked as reproduced baselines')
    model=SegmentationCNN(base=CFG.base_channels,proposed=proposed).to(DEVICE)
    optimizer=torch.optim.AdamW(model.parameters(),lr=CFG.lr,weight_decay=CFG.weight_decay)
    scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode='max',factor=0.5,patience=3)
    scaler=torch.amp.GradScaler('cuda',enabled=(DEVICE.type=='cuda'))
    train_data=PolypDataset(train_table,CFG.image_size)
    epoch=0; next_batch=0; loss_sum=0.; sample_count=0; patience_counter=0; best_dice=-1.
    run_signature={'spec':spec,'seed':seed,'config':asdict(CFG),'manifest_id':manifest_id}
    if latest.exists():
        # Only load local checkpoints created by this notebook; arbitrary pickle payloads are unsafe.
        state=torch.load(latest,map_location=DEVICE,weights_only=False)
        if state['signature']!=run_signature:
            raise ValueError('Checkpoint config/data mismatch; refuse unsafe resume. Use new run folder or exact old configuration.')
        model.load_state_dict(state['model']);optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler']);scaler.load_state_dict(state['scaler'])
        epoch=state['epoch'];next_batch=state['next_batch'];loss_sum=state['loss_sum'];sample_count=state['sample_count']
        patience_counter=state['patience_counter'];best_dice=state['best_dice']
        restore_rng(state['rng'])
        print('AUTO-RESUME',run_name,'epoch=',epoch,'next_batch=',next_batch,'best Dice=',round(best_dice,4))
    else:
        print('NEW RUN',run_name)
    def checkpoint(path, ep, nb, loss_total, n_samples):
        save_atomic(path,{
            'signature':run_signature,'model':model.state_dict(),'optimizer':optimizer.state_dict(),
            'scheduler':scheduler.state_dict(),'scaler':scaler.state_dict(),'epoch':ep,
            'next_batch':nb,'loss_sum':loss_total,'sample_count':n_samples,
            'best_dice':best_dice,'patience_counter':patience_counter,'rng':capture_rng()})
    num_batches=math.ceil(len(train_data)/CFG.batch_size)
    while epoch < CFG.max_epochs:
        model.train()
        ordered=np.random.default_rng(seed + epoch*10007).permutation(len(train_data)).tolist()
        all_batches=[ordered[i:i+CFG.batch_size] for i in range(0,len(ordered),CFG.batch_size)]
        remaining_batches=all_batches[next_batch:]
        # No shuffle, no prefetch: already processed batches will not be loaded/augmented.
        loader=DataLoader(train_data,batch_sampler=remaining_batches,num_workers=0)
        for batch_index,(x,y,source_ids) in enumerate(loader,start=next_batch):
            x,y=x.to(DEVICE),y.to(DEVICE)
            source_ids=source_ids.to(DEVICE)
            x,y=augment_aligned(x,y)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda',enabled=(DEVICE.type=='cuda')):
                logits,edge=model(x,source_ids,update_domain_stats=proposed)
                loss=segmentation_loss(logits,y)
                if proposed:
                    loss=loss+CFG.boundary_weight*F.binary_cross_entropy_with_logits(edge,boundary_mask(y))
                    if batch_index%CFG.consistency_every_batches==0:
                        # First view is detached, second view receives gradient.
                        logits_alt,_=model(style_shift(x))
                        loss=loss+CFG.consistency_weight*F.mse_loss(torch.sigmoid(logits_alt),torch.sigmoid(logits).detach())
            if not torch.isfinite(loss):raise FloatingPointError(f'Nonfinite loss at {run_name} epoch={epoch} batch={batch_index}')
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.)
            scaler.step(optimizer);scaler.update()
            n=x.size(0);loss_sum+=float(loss.detach().cpu())*n;sample_count+=n
            if (batch_index+1)%CFG.checkpoint_every_batches==0:
                checkpoint(latest,epoch,batch_index+1,loss_sum,sample_count)
        val_score=evaluate_model(model,val_table)
        scheduler.step(val_score['dice'])
        improved=val_score['dice']>best_dice+1e-5
        if improved:
            best_dice=val_score['dice'];patience_counter=0
        else:patience_counter+=1
        record={'epoch':epoch+1,'train_loss':loss_sum/max(sample_count,1),
                'val_dice':val_score['dice'],'val_iou':val_score['iou'],
                'best_dice':best_dice,'lr':optimizer.param_groups[0]['lr'],'seed':seed,'run':run_name}
        # Save before recording history; checkpoint restores correct epoch and early-stopping state.
        checkpoint(latest,epoch+1,0,0.,0)
        if improved:checkpoint(best,epoch+1,0,0.,0)
        # Duplicate history line possible if killed after checkpoint save, before history write; remove duplicates downstream.
        with open(history_file,'a',encoding='utf-8') as log:log.write(json.dumps(record)+'\n')
        print(f"{run_name} epoch {epoch+1}/{CFG.max_epochs}: loss={record['train_loss']:.4f} valDice={val_score['dice']:.4f} best={best_dice:.4f} patience={patience_counter}")
        if patience_counter>=CFG.early_stop_patience:
            print('EARLY STOP:',run_name)
            break
        epoch+=1;next_batch=0;loss_sum=0.;sample_count=0
    return best

## 5. Train → validate → test independently

**Predefined controls**: U-Net trained on Kvasir only; U-Net trained on mixed Kvasir+ClinicDB; DCRB-Net trained on the same mixed data. The comparison isolates (a) effect of combining datasets and (b) architectural/regularization changes. The primary paper's specific **nine published algorithms must be added after its full methodology is obtained**. The Sushama & Menon (2024) algorithm is *not* silently replaced by generic U-Net.

**Strong-novelty ablation required before submission**: remove source-statistic calibration; remove reliability gate; remove boundary gate/head; remove style consistency; full DCRB-Net, all under same splits/seed/budget. Also compare to competitive recent architectures using verified official code and identical data partitions.

In [7]:
if RUN_EXPERIMENTS:
    assert manifest is not None
    for seed in SEEDS:
        for spec in EXPERIMENTS:
            best_checkpoint=train_one(spec,seed,manifest)
            print('Saved best checkpoint:',best_checkpoint)
else:
    print('Training intentionally disabled. Download/restructure verified datasets and set RUN_EXPERIMENTS=True.')

Training intentionally disabled. Download/restructure verified datasets and set RUN_EXPERIMENTS=True.


In [8]:
def load_best_model(architecture, checkpoint_file):
    proposed=(architecture=='dcrb')
    model=SegmentationCNN(base=CFG.base_channels,proposed=proposed).to(DEVICE)
    ckpt=torch.load(checkpoint_file,map_location=DEVICE,weights_only=False)
    model.load_state_dict(ckpt['model'])
    return model.eval()

if RUN_EXPERIMENTS:
    ext=manifest.loc[manifest.split.eq('external_test')].reset_index(drop=True)
    for seed in SEEDS:
        for spec in EXPERIMENTS:
            run_name=f"{spec['name']}_seed{seed}"
            best_file=OUTPUT_ROOT/run_name/'checkpoint_best.pt'
            if not best_file.exists():
                print('SKIP NO BEST CHECKPOINT:',run_name);continue
            model=load_best_model(spec['architecture'],best_file)
            table=evaluate_model(model,ext,return_rows=True)
            table['run']=run_name;table['seed']=seed
            table.to_csv(OUTPUT_ROOT/run_name/'external_test_per_image.csv',index=False)
            print(run_name,'EXTERNAL TEST:',table.groupby('source')[['dice','iou']].mean().round(4).to_dict('index'))
            del model
else:
    print('External test evaluation disabled in smoke mode')

External test evaluation disabled in smoke mode


## 6. Ablation protocol & reporting checklist

1. **Reproduce the primary paper honestly**: obtain the full published methods/PDF or official source code; replicate its exact dataset combinations, nine architectures, splits, preprocessing, training budgets, and metrics. If unavailable, label our multi-dataset U-Net comparison as a *paper-inspired control*, never as the original implementation.
2. **Companion comparison**: obtain Sushama & Menon (2024) full protocol/code and implement separately; published metrics under different splits are only contextual, not a fair head-to-head.
3. **Fixed split and source definitions**: verify mask-image alignment, duplicate/patient/video leakage and licensing. Freeze an immutable manifest with checksums before final experiments.
4. **Ablation**: U-Net, U-Net + reliability only, U-Net + boundary only, U-Net + consistency only, full DCRB-Net. The v0.1 prototype implements the endpoints; intermediate ablation configurations require parameterized modules before claiming the ablation complete.
5. **At least three seeds** under identical settings (e.g. 42/123/2025); mean ± SD, 95% CI when justifiable, Dice, IoU, boundary F1, HD95 if applicable; parameter count, FLOPs, FPS, peak CUDA memory.
6. **Fairness**: identical train/val/test splits, input size, augmentation and search budget; no target-test tuning. Compare training regimes separately rather than comparing a domain-generalization method to a target-adapted method without explanation.
7. **Checkpointing and resume**: `checkpoint_latest.pt` saved atomically every N processed batches; `checkpoint_best.pt` for best validation Dice; both include full state; early stopping uses validation only. An epoch interrupted before its batch checkpoint must repeat at most those N batches.
8. **Manuscript claims**: no invented numbers or unsupported SOTA claims; distinguish pilot and publication-grade experimental results; no implied clinical readiness.

### Current verification status
- Verified notebook architecture tensor shapes: via synthetic forward pass in this notebook.
- Training **not executed** against public datasets: datasets are not supplied; download and audit are prerequisites.
- The proposed module is an experimental candidate pending novelty and empirical validation.